<a href="https://colab.research.google.com/github/Aaricis/Hung-yi-Lee-ML2022/blob/main/HW13/ml2022spring_hw13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 思路

## Simple Baseline (acc > 0.44820)

Score: 0.52191 Private score: 0.51344

运行助教代码。

## Medium Baseline (acc > 0.64840)

Score: 0.66832 Private score: 0.66111

实现knowledge distillation并训练更多epoch。
- KL divergence loss function


```python
# Implement the loss function with KL divergence loss for knowledge distillation.
# You also have to copy-paste this whole block to HW13 GradeScope.
def loss_fn_kd(student_logits, labels, teacher_logits, alpha=0.5, temperature=1.0):
    # ------------TODO-------------
    # Refer to the above formula and finish the loss function for knowkedge distillation using KL divergence loss and CE loss.
    # If you have no idea, please take a look at the provided useful link above.
    kl_loss = nn.KLDivLoss(reduction='batchmean', log_target= True)
    ce_loss = nn.CrossEntropyLoss()
    
    log_softmax = nn.LogSoftmax(dim=1)  # 定义LogSoftmax
    p = log_softmax(student_logits / temperature)
    q = log_softmax(teacher_logits / temperature)
    
    loss = alpha * (temperature ** 2) * kl_loss(p, q) + (1 - alpha) * ce_loss(student_logits, labels)
    return loss
```
- 训练50个epoch

```python
'n_epochs': 50
```







## Strong Baseline (acc > 0.82370)

Score: 0.84462 Private score: 0.81263

实现Depthwise&Pointwise Convolution（深度可分离卷积）、对齐Teacher模型中间层特征表示、训练更多epoch、Training阶段使用Data Augmentation、Inference阶段使用Test Time Augmentation。

- 参照[MobileNet](https://arxiv.org/abs/1704.04861)论文，实现使用depthwise and pointwise convolution的Student模型架构；


```python
# For Strong Baseline

def dwpw_conv(in_channels, out_channels, kernel_size, stride=1, padding=1,bias=False):
    return nn.Sequential(
        nn.Conv2d(in_channels, in_channels, kernel_size, stride=stride, padding=padding,bias=bias, groups=in_channels), #depthwise convolution
        nn.BatchNorm2d(in_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(in_channels, out_channels, 1,  bias= bias,), # pointwise convolution
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    )

class StudentNet(nn.Module):
    def __init__(self, inplanes = 64):
        super().__init__()
        self.inplanes = inplanes
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(self.inplanes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = dwpw_conv(inplanes, inplanes, kernel_size=3)
        self.layer2 = dwpw_conv(inplanes, 128, kernel_size=3, stride=2)
        self.layer3 = dwpw_conv(128, 256, kernel_size=3, stride=2)
        self.layer4 = dwpw_conv(256, 141, kernel_size=3, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(141, 11)

    def forward(self, x):
        x=self.conv1(x)
        x=self.bn1(x)
        x=self.relu(x)
        x=self.maxpool(x)

        x=self.layer1(x)
        x=self.layer2(x)
        x=self.layer3(x)
        x=self.layer4(x)

        x=self.avgpool(x)
        x = torch.flatten(x, 1)
        x=self.fc(x)

        return x

def get_student_model(): # This function should have no arguments so that we can get your student network by directly calling it.
    # you can modify or do anything here, just remember to return an nn.Module as your student network.
    return StudentNet()
```
- 对齐Teacher模型中间层特征表示；

助教提供的Teacher模型为resnet18，我们可以查看该模型的详细信息。根据resnet18的模型结构，我们精心设计Student模型一些层的名称、类型和参数形状与Teacher模型一致。在训练过程中引入中间层的特征损失，约束Student网络的中间层特征与Teacher网络的对应层特征分布相似，实现知识迁移。相比仅对齐最终输出，中间层对齐能保留更多结构化知识。

**在实作中，如何获得模型中间层的输出呢？**

> 通过PyTorch的forward hook机制，捕获Teacher模型（ResNet18）和Student模型（自定义StudentNet）在推理过程中指定中间层的输出特征。


在实作中，我们对齐两个模型`layer1`、`layer2`、`layer3`的输出特征表示。

```python
Slayer1out, Slayer2out, Slayer3out, Tlayer1out, Tlayer2out, Tlayer3out = [], [], [], [], [], []

def hookS1(module, input, output):
  Slayer1out.append(output)
  return None

def hookS2(module, input, output):
  Slayer2out.append(output)
  return None

def hookS3(module, input, output):
  Slayer3out.append(output)
  return None

def hookT1(module, input, output):
  Tlayer1out.append(output)
  return None

def hookT2(module, input, output):
  Tlayer2out.append(output)
  return None

def hookT3(module, input, output):
  Tlayer3out.append(output)
  return None

student_model.layer1.register_forward_hook(hookS1)
student_model.layer2.register_forward_hook(hookS2)
student_model.layer3.register_forward_hook(hookS3)

teacher_model.layer1.register_forward_hook(hookT1)
teacher_model.layer2.register_forward_hook(hookT2)
teacher_model.layer3.register_forward_hook(hookT3)
```

损失函数在Medium Baseline基础上，增加`loss_hidden`，学习中间层输出的差异。

```python
loss_hidden = F.smooth_l1_loss(slayer1out, tlayer1out) + F.smooth_l1_loss(slayer2out, tlayer2out) + F.smooth_l1_loss(slayer3out, tlayer3out)
```
完整损失函数为：

```python
loss =  loss_hidden + 10 * lamb * loss_output
```
`lamb`为权重系数，随着训练的加深动态增长。
```python
p=epoch/(n_epochs-1)
lamb= p * p # 0-1
```

- 训练300个epoch，当训练50个epoch没有提升时，终止训练；

```python
'n_epochs': 300
'patience': 50
```
- Data Augmentation & Test Time Augmentation；

与HW3保持一致，详情参见HW3。

[ML2022-HW3-Image Classification](https://zhuanlan.zhihu.com/p/28149430319)


## Boss Baseline (acc > 0.85159)

Score: 0.86254 Private score: 0.84379

在Strong Baseline中，使用的Data Augmentation进行过度的增强反而可能损害性能。将`train_tfm`还原为默认设置，训练300个epoch，最后ensemble多份结果。


# Homework 13 - Network Compression

Author: Liang-Hsuan Tseng (b07502072@ntu.edu.tw), modified from ML2021-HW13

If you have any questions, feel free to ask: ntu-ml-2022spring-ta@googlegroups.com

[**Link to HW13 Slides**](https://docs.google.com/presentation/d/1nCT9XrInF21B4qQAWuODy5sonKDnpGhjtcAwqa75mVU/edit#slide=id.p)

## Outline

* [Packages](#Packages) - intall some required packages.
* [Dataset](#Dataset) - something you need to know about the dataset.
* [Configs](#Configs) - the configs of the experiments, you can change some hyperparameters here.
* [Architecture_Design](#Architecture_Design) - depthwise and pointwise convolution examples and some useful links.
* [Knowledge_Distillation](#Knowledge_Distillation) - KL divergence loss for knowledge distillation and some useful links.
* [Training](#Training) - training loop implementation modified from HW3.
* [Inference](#Inference) - create submission.csv by using the student_best.ckpt from the previous experiment.



### Packages
First, we need to import some useful packages. If the torchsummary package are not intalled, please install it via `pip install torchsummary`

In [ ]:
# Import some useful packages for this homework
import numpy as np
import pandas as pd
import torch
import os
import torch.nn as nn
import torch.nn.functional as F
# import torchvision.transforms as transforms
import torchvision.transforms.v2 as transforms
from PIL import Image
from torch.utils.data import ConcatDataset, DataLoader, Subset, Dataset # "ConcatDataset" and "Subset" are possibly useful
from torchvision.datasets import DatasetFolder, VisionDataset
from torchsummary import summary
from tqdm.auto import tqdm
import random

from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# !nvidia-smi # list your current GPU

### Configs
In this part, you can specify some variables and hyperparameters as your configs.

In [ ]:
cfg = {
    'dataset_root': './food11-hw13',
    'save_dir': './outputs',
    'exp_name': 'strong_baseline', #"simple_baseline",
    'batch_size': 64,
    'lr': 3e-4,
    'seed': 20220013,
    'loss_fn_type': 'KD', #'CE', # simple baseline: CE, medium baseline: KD. See the Knowledge_Distillation part for more information.
    'weight_decay': 1e-5,
    'grad_norm_max': 10, # 最大允许的梯度范数（阈值）
    'n_epochs': 300, #50, #10, # train more steps to pass the medium baseline.
    'patience': 50, #300,
}

In [ ]:
myseed = cfg['seed']  # set a random seed for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(myseed)
torch.manual_seed(myseed)
random.seed(myseed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(myseed)

save_path = os.path.join(cfg['save_dir'], cfg['exp_name']) # create saving directory
os.makedirs(save_path, exist_ok=True)

# define simple logging functionality
log_fw = open(f"{save_path}/log.txt", 'w') # open log file to save log outputs
def log(text):     # define a logging function to trace the training process
    print(text)
    log_fw.write(str(text)+'\n')
    log_fw.flush()

log(cfg)  # log your configs to the log file

{'dataset_root': './food11-hw13', 'save_dir': './outputs', 'exp_name': 'strong_baseline', 'batch_size': 64, 'lr': 0.0003, 'seed': 20220013, 'loss_fn_type': 'KD', 'weight_decay': 1e-05, 'grad_norm_max': 10, 'n_epochs': 300, 'patience': 50}


### Dataset
We use Food11 dataset for this homework, which is similar to homework3. But remember, Please DO NOT utilize the dataset of HW3. We've modified the dataset, so you should only access the dataset by loading it in this kaggle notebook or through the links provided in the HW13 colab notebooks.

In [ ]:
# fetch and download the dataset from github (about 1.12G)
# !wget https://github.com/virginiakm1988/ML2022-Spring/raw/main/HW13/food11-hw13.tar.gz
## backup links:

!wget https://github.com/andybi7676/ml2022spring-hw13/raw/main/food11-hw13.tar.gz -O food11-hw13.tar.gz
# !gdown '1ijKoNmpike_yjUw8SWRVVWVoMOXXqycj' --output food11-hw13.tar.gz

--2025-06-10 10:17:12--  https://github.com/andybi7676/ml2022spring-hw13/raw/main/food11-hw13.tar.gz
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://media.githubusercontent.com/media/andybi7676/ml2022spring-hw13/main/food11-hw13.tar.gz [following]
--2025-06-10 10:17:13--  https://media.githubusercontent.com/media/andybi7676/ml2022spring-hw13/main/food11-hw13.tar.gz
Resolving media.githubusercontent.com (media.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to media.githubusercontent.com (media.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1203320552 (1.1G) [application/octet-stream]
Saving to: ‘food11-hw13.tar.gz’

food11-hw13.tar.gz  100%[===================>]   1.12G   211MB/s    in 8.5s    

2025-06-10 10:19:28 (136 MB/s) - ‘food11-

In [ ]:
# extract the data
!tar -xzf ./food11-hw13.tar.gz # Could take some time
# !tar -xzvf ./food11-hw13.tar.gz # use this command if you want to checkout the whole process.

In [ ]:
for dirname, _, filenames in os.walk('./food11-hw13'):
    if len(filenames) > 0:
        print(f"{dirname}: {len(filenames)} files.") # Show the file amounts in each split.

./food11-hw13: 1 files.
./food11-hw13/evaluation: 3347 files.
./food11-hw13/validation: 3430 files.
./food11-hw13/training: 9866 files.


Next, specify train/test transform for image data augmentation.
Torchvision provides lots of useful utilities for image preprocessing, data wrapping as well as data augmentation.

Please refer to [PyTorch official website](https://pytorch.org/vision/stable/transforms.html) for details about different transforms. You can also apply the knowledge or experience you learned in HW3.

In [ ]:
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# define training/testing transforms
test_tfm = transforms.Compose([
    # It is not encouraged to modify this part if you are using the provided teacher model. This transform is stardard and good enough for testing.
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    normalize,
])

# train_tfm = transforms.Compose([
#     # add some useful transform or augmentation here, according to your experience in HW3.
#     transforms.Resize(256),  # You can change this
#     transforms.CenterCrop(224), # You can change this, but be aware of that the given teacher model's input size is 224.
#     # The training input size of the provided teacher model is (3, 224, 224).
#     # Thus, Input size other then 224 might hurt the performance. please be careful.
#     transforms.RandomHorizontalFlip(), # You can change this.
#     transforms.ToTensor(),
#     normalize,
# ])

# train_tfm = transforms.Compose([
#     transforms.RandomResizedCrop(224, antialias=True),
#     transforms.RandomHorizontalFlip(0.5),
#     transforms.TrivialAugmentWide(),
#     transforms.PILToTensor(),
#     transforms.ConvertImageDtype(torch.float),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
#  ])

# For Strong Baseline
train_tfm = transforms.Compose([
    # Resize the image into a fixed shape (height = width = 128)
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    # You may add some transforms here.
    # Flip the image horizontally with a probability of 20%.
    transforms.RandomHorizontalFlip(p=0.2),
    # Flip the image vertically with a probability of 10%.
    transforms.RandomVerticalFlip(p=0.1),
    # 20% probability for gray scale change
    transforms.RandomGrayscale(0.2),
    # Randomly adjust the brightness, contrast, saturation and hue of the image.
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    # Randomly rotate the image within the range of -10 to 10 degrees.
    transforms.RandomRotation(degrees=(-10, 10)),
    # ToTensor() should be the last one of the transforms.
    transforms.ToTensor(),
    normalize,
])

/usr/local/lib/python3.11/dist-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [ ]:
from torch.utils.data import default_collate

NUM_CLASSES = 11
cutmix = transforms.CutMix(num_classes= NUM_CLASSES)
mixup = transforms.MixUp(num_classes= NUM_CLASSES)
cutmix_or_mixup = transforms.RandomChoice([cutmix, mixup])

def collate_fn(batch):
    return cutmix_or_mixup(*default_collate(batch))

In [ ]:
class FoodDataset(Dataset):
    def __init__(self, path, tfm=test_tfm, files = None):
        super().__init__()
        self.path = path
        self.files = sorted([os.path.join(path,x) for x in os.listdir(path) if x.endswith(".jpg")])
        if files != None:
            self.files = files
        print(f"One {path} sample",self.files[0])
        self.transform = tfm

    def __len__(self):
        return len(self.files)

    def __getitem__(self,idx):
        fname = self.files[idx]
        im = Image.open(fname)
        im = self.transform(im)
        try:
            label = int(fname.split("/")[-1].split("_")[0])
        except:
            label = -1 # test has no label
        return im,label

In [ ]:
# Form train/valid dataloaders
train_set = FoodDataset(os.path.join(cfg['dataset_root'],"training"), tfm=train_tfm)
# train_loader = DataLoader(train_set, batch_size=cfg['batch_size'], shuffle=True, num_workers=0, pin_memory=True)
train_loader = DataLoader(train_set, batch_size=cfg['batch_size'], shuffle=True, num_workers=0, pin_memory=True, drop_last=True, collate_fn=collate_fn) # CutMix and MixUp Transform


valid_set = FoodDataset(os.path.join(cfg['dataset_root'], "validation"), tfm=test_tfm)
valid_loader = DataLoader(valid_set, batch_size=cfg['batch_size'], shuffle=False, num_workers=0, pin_memory=True)

One ./food11-hw13/training sample ./food11-hw13/training/0_0.jpg
One ./food11-hw13/validation sample ./food11-hw13/validation/0_0.jpg


### Architecture_Design

In this homework, you have to design a smaller network and make it perform well. Apparently, a well-designed architecture is crucial for such task. Here, we introduce the depthwise and pointwise convolution. These variants of convolution are some common techniques for architecture design when it comes to network compression.

<img src="https://i.imgur.com/LFDKHOp.png" width=400px>

* explanation of depthwise and pointwise convolutions:
    * [prof. Hung-yi Lee's slides(p.24~p.30, especially p.28)](https://speech.ee.ntu.edu.tw/~hylee/ml/ml2021-course-data/tiny_v7.pdf)

In [ ]:
# Example implementation of Depthwise and Pointwise Convolution
def dwpw_conv(in_channels, out_channels, kernel_size, stride=1, padding=0):
    return nn.Sequential(
        nn.Conv2d(in_channels, in_channels, kernel_size, stride=stride, padding=padding, groups=in_channels), #depthwise convolution
        nn.Conv2d(in_channels, out_channels, 1), # pointwise convolution
    )

* other useful techniques
    * [group convolution](https://www.researchgate.net/figure/The-transformations-within-a-layer-in-DenseNets-left-and-CondenseNets-at-training-time_fig2_321325862) (Actually, depthwise convolution is a specific type of group convolution)
    * [SqueezeNet](https://arxiv.org/abs/1602.07360)
    * [MobileNet](https://arxiv.org/abs/1704.04861)
    * [ShuffleNet](https://arxiv.org/abs/1707.01083)
    * [Xception](https://arxiv.org/abs/1610.02357)
    * [GhostNet](https://arxiv.org/abs/1911.11907)


After introducing depthwise and pointwise convolutions, let's define the **student network architecture**. Here, we have a very simple network formed by some regular convolution layers and pooling layers. You can replace the regular convolution layers with the depthwise and pointwise convolutions. In this way, you can further increase the depth or the width of your network architecture.

In [ ]:
# # Define your student network here. You have to copy-paste this code block to HW13 GradeScope before deadline.
# # We will use your student network definition to evaluate your results(including the total parameter amount).

# class StudentNet(nn.Module):
#     def __init__(self):
#       super().__init__()

#       # ---------- TODO ----------
#       # Modify your model architecture

#       self.cnn = nn.Sequential(
#         nn.Conv2d(3, 32, 3),
#         nn.BatchNorm2d(32),
#         nn.ReLU(),
#         nn.Conv2d(32, 32, 3),
#         nn.BatchNorm2d(32),
#         nn.ReLU(),
#         nn.MaxPool2d(2, 2, 0),

#         nn.Conv2d(32, 64, 3),
#         nn.BatchNorm2d(64),
#         nn.ReLU(),
#         nn.MaxPool2d(2, 2, 0),

#         nn.Conv2d(64, 100, 3),
#         nn.BatchNorm2d(100),
#         nn.ReLU(),
#         nn.MaxPool2d(2, 2, 0),

#         # Here we adopt Global Average Pooling for various input size.
#         nn.AdaptiveAvgPool2d((1, 1)),
#       )
#       self.fc = nn.Sequential(
#         nn.Linear(100, 11),
#       )

#     def forward(self, x):
#       out = self.cnn(x)
#       out = out.view(out.size()[0], -1)
#       return self.fc(out)

# def get_student_model(): # This function should have no arguments so that we can get your student network by directly calling it.
#     # you can modify or do anything here, just remember to return an nn.Module as your student network.
#     return StudentNet()

# # End of definition of your student model and the get_student_model API
# # Please copy-paste the whole code block, including the get_student_model function.

In [ ]:
# For Strong Baseline

def dwpw_conv(in_channels, out_channels, kernel_size, stride=1, padding=1,bias=False):
    return nn.Sequential(
        nn.Conv2d(in_channels, in_channels, kernel_size, stride=stride, padding=padding,bias=bias, groups=in_channels), #depthwise convolution
        nn.BatchNorm2d(in_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(in_channels, out_channels, 1,  bias= bias,), # pointwise convolution
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    )

class StudentNet(nn.Module):
    def __init__(self, inplanes = 64):
        super().__init__()
        self.inplanes = inplanes
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(self.inplanes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = dwpw_conv(inplanes, inplanes, kernel_size=3)
        self.layer2 = dwpw_conv(inplanes, 128, kernel_size=3, stride=2)
        self.layer3 = dwpw_conv(128, 256, kernel_size=3, stride=2)
        self.layer4 = dwpw_conv(256, 141, kernel_size=3, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(141, 11)

    def forward(self, x):
        x=self.conv1(x)
        x=self.bn1(x)
        x=self.relu(x)
        x=self.maxpool(x)

        x=self.layer1(x)
        x=self.layer2(x)
        x=self.layer3(x)
        x=self.layer4(x)

        x=self.avgpool(x)
        x = torch.flatten(x, 1)
        x=self.fc(x)

        return x

def get_student_model(): # This function should have no arguments so that we can get your student network by directly calling it.
    # you can modify or do anything here, just remember to return an nn.Module as your student network.
    return StudentNet()

After specifying the student network architecture, please use `torchsummary` package to get information about the network and verify the total number of parameters. Note that the total params of your student network should not exceed the limit (`Total params` in `torchsummary` ≤ 100,000).

In [ ]:
# DO NOT modify this block and please make sure that this block can run sucessfully.
student_model = get_student_model()
summary(student_model, (3, 224, 224), device='cpu')
# You have to copy&paste the results of this block to HW13 GradeScope.

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 112, 112]           9,408
       BatchNorm2d-2         [-1, 64, 112, 112]             128
              ReLU-3         [-1, 64, 112, 112]               0
         MaxPool2d-4           [-1, 64, 56, 56]               0
            Conv2d-5           [-1, 64, 56, 56]             576
       BatchNorm2d-6           [-1, 64, 56, 56]             128
              ReLU-7           [-1, 64, 56, 56]               0
            Conv2d-8           [-1, 64, 56, 56]           4,096
       BatchNorm2d-9           [-1, 64, 56, 56]             128
             ReLU-10           [-1, 64, 56, 56]               0
           Conv2d-11           [-1, 64, 28, 28]             576
      BatchNorm2d-12           [-1, 64, 28, 28]             128
             ReLU-13           [-1, 64, 28, 28]               0
           Conv2d-14          [-1, 128,

In [ ]:
# Load provided teacher model (model architecture: resnet18, num_classes=11, test-acc ~= 89.9%)
teacher_model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', pretrained=False, num_classes=11)
# load state dict
teacher_ckpt_path = os.path.join(cfg['dataset_root'], "resnet18_teacher.ckpt")
teacher_model.load_state_dict(torch.load(teacher_ckpt_path, map_location='cpu'))
# Now you already know the teacher model's architecture. You can take advantage of it if you want to pass the strong or boss baseline.
# Source code of resnet in pytorch: (https://github.com/pytorch/vision/blob/main/torchvision/models/resnet.py)
# You can also see the summary of teacher model. There are 11,182,155 parameters totally in the teacher model
# summary(teacher_model, (3, 224, 224), device='cpu')

Downloading: "https://github.com/pytorch/vision/zipball/v0.10.0" to /root/.cache/torch/hub/v0.10.0.zip
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


<All keys matched successfully>

### 中间层对齐（Intermediate Layer Alignment）

In [ ]:
Slayer1out, Slayer2out, Slayer3out, Tlayer1out, Tlayer2out, Tlayer3out = [], [], [], [], [], []

def hookS1(module, input, output):
  Slayer1out.append(output)
  return None

def hookS2(module, input, output):
  Slayer2out.append(output)
  return None

def hookS3(module, input, output):
  Slayer3out.append(output)
  return None

def hookT1(module, input, output):
  Tlayer1out.append(output)
  return None

def hookT2(module, input, output):
  Tlayer2out.append(output)
  return None

def hookT3(module, input, output):
  Tlayer3out.append(output)
  return None

student_model.layer1.register_forward_hook(hookS1)
student_model.layer2.register_forward_hook(hookS2)
student_model.layer3.register_forward_hook(hookS3)

teacher_model.layer1.register_forward_hook(hookT1)
teacher_model.layer2.register_forward_hook(hookT2)
teacher_model.layer3.register_forward_hook(hookT3)

In [ ]:
def use_pretrain():
  student_model.conv1.weight = teacher_model.conv1.weight
  student_model.bn1.weight = teacher_model.bn1.weight
  student_model.bn1.bias = teacher_model.bn1.bias
  student_model.bn1.running_mean = teacher_model.bn1.running_mean
  student_model.bn1.running_var = teacher_model.bn1.running_var
  student_model.conv1.weight.requires_grad = False
  student_model.bn1.weight.requires_grad = False
  student_model.bn1.bias.requires_grad = False

use_pretrain()

### Knowledge_Distillation

<img src="https://i.imgur.com/H2aF7Rv.png=100x" width="400px">

Since we have a learned big model, let it teach the other small model. In implementation, let the training target be the prediction of big model instead of the ground truth.

**Why it works?**
* If the data is not clean, then the prediction of big model could ignore the noise of the data with wrong labeled.
* There might have some relations between classes, so soft labels from teacher model might be useful. For example, Number 8 is more similar to 6, 9, 0 than 1, 7.


**How to implement?**
* $Loss = \alpha T^2 \times KL(p || q) + (1-\alpha)(\text{Original Cross Entropy Loss}), \text{where } p=softmax(\frac{\text{student's logits}}{T}), \text{and } q=softmax(\frac{\text{teacher's logits}}{T})$
* very useful link: [pytorch docs of KLDivLoss with examples](https://pytorch.org/docs/stable/generated/torch.nn.KLDivLoss.html)
* original paper: [Distilling the Knowledge in a Neural Network](https://arxiv.org/abs/1503.02531)

In [ ]:
# Implement the loss function with KL divergence loss for knowledge distillation.
# You also have to copy-paste this whole block to HW13 GradeScope.
def loss_fn_kd(student_logits, labels, teacher_logits, alpha=0.5, temperature=1.0):
    # ------------TODO-------------
    # Refer to the above formula and finish the loss function for knowkedge distillation using KL divergence loss and CE loss.
    # If you have no idea, please take a look at the provided useful link above.
    kl_loss = nn.KLDivLoss(reduction='batchmean', log_target= True)
    ce_loss = nn.CrossEntropyLoss(label_smoothing=0.1)

    log_softmax = nn.LogSoftmax(dim=1)  # 定义LogSoftmax
    p = log_softmax(student_logits / temperature)
    q = log_softmax(teacher_logits / temperature)

    loss = alpha * (temperature ** 2) * kl_loss(p, q) + (1 - alpha) * ce_loss(student_logits, labels)
    return loss

In [ ]:
# boss
# def pairwise_distance(x):
#     """Calculate pairwise distance between batch samples."""
#     return torch.cdist(x, x, p=2)

# def angle_between_pairs(x):
#     """Calculate angles between all pairs of points in batch."""
#     diff = x.unsqueeze(1) - x.unsqueeze(0)
#     norm = diff.norm(dim=-1, p=2, keepdim=True)
#     normalized_diff = diff / (norm + 1e-8)
#     angles = torch.bmm(normalized_diff, normalized_diff.transpose(1, 2))
#     return angles

def pairwise_distance(x):
    """计算带对数变换的成对距离"""
    dist = torch.cdist(x, x, p=2)
    return torch.log(1 + dist)  # 稳定化处理

def angle_between_pairs(x):
    """计算余弦相似度（替代角度）"""
    x_norm = F.normalize(x, p=2, dim=-1)
    return torch.mm(x_norm, x_norm.t())

def loss_fn_rkd(teacher_feature, student_feature, labels, alpha=0.5):
    """
    Relational Knowledge Distillation Loss Function.

    Args:
    - teacher_feature: Teacher model feature embeddings.
    - student_feature: Student model feature embeddings.
    - labels: Ground truth labels.
    - alpha: Weighting factor for relational distillation loss.

    Returns:
    - loss: Combined relational knowledge and hard label loss.
    """

    # Pairwise distances between features in the teacher and student model
    teacher_dist = pairwise_distance(teacher_feature)
    student_dist = pairwise_distance(student_feature)

    # Distillation loss using the L2 norm between relational distances
    distance_loss = F.mse_loss(student_dist, teacher_dist)

    # Angle-based loss between teacher and student feature vectors
    teacher_angle = angle_between_pairs(teacher_feature)
    student_angle = angle_between_pairs(student_feature)
    angle_loss = F.mse_loss(student_angle, teacher_angle)

    # Hard label cross-entropy loss for the student output
    # hard_loss = F.cross_entropy(student_feature, labels)

    # Combine the losses
    # loss = alpha * hard_loss + (1 - alpha) * (distance_loss + angle_loss)
    loss = alpha * (distance_loss + angle_loss)
    return loss

In [ ]:
# boss
def loss_fn_dm(teacher_feature, student_feature, labels, alpha=0.5):
    """
    Distance Metric (DM) Knowledge Distillation Loss Function.

    Args:
    - teacher_feature: The feature representations from the teacher model.
    - student_feature: The feature representations from the student model.
    - labels: Ground truth labels for the task.
    - alpha: Weighting factor for distance metric loss.

    Returns:
    - loss: Combined distance metric loss and cross-entropy loss.
    """
    # Calculate pairwise distance between teacher and student embeddings
    teacher_dist = pairwise_distance(teacher_feature)
    student_dist = pairwise_distance(student_feature)

    # Distance metric loss using Mean Squared Error (MSE) loss
    dist_loss = F.mse_loss(student_dist, teacher_dist)

    # Hard label cross-entropy loss for the student's output
    # hard_loss = F.cross_entropy(student_feature, labels)

    # Combine the losses
    # loss = alpha * hard_loss + (1 - alpha) * dist_loss
    loss = alpha * dist_loss
    return loss

In [ ]:
# choose the loss function by the config
if cfg['loss_fn_type'] == 'CE':
    # For the classification task, we use cross-entropy as the default loss function.
    loss_fn = nn.CrossEntropyLoss() # loss function for simple baseline.

if cfg['loss_fn_type'] == 'KD': # KD stands for knowledge distillation
    loss_fn = loss_fn_kd # implement loss_fn_kd for the report question and the medium baseline.

# You can also adopt other types of knowledge distillation techniques for strong and boss baseline, but use function name other than `loss_fn_kd`
# For example:
# def loss_fn_custom_kd():
#     pass
# if cfg['loss_fn_type'] == 'custom_kd':
#     loss_fn = loss_fn_custom_kd

# "cuda" only when GPUs are available.
device = "cuda" if torch.cuda.is_available() else "cpu"
log(f"device: {device}")

# The number of training epochs and patience.
n_epochs = cfg['n_epochs']
patience = cfg['patience'] # If no improvement in 'patience' epochs, early stop

device: cuda


### Training
implement training loop for simple baseline, feel free to modify it.

In [ ]:
# # Initialize a model, and put it on the device specified.
# student_model.to(device)
# teacher_model.to(device) # MEDIUM BASELINE

# # Initialize optimizer, you may fine-tune some hyperparameters such as learning rate on your own.
# optimizer = torch.optim.Adam(student_model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])

# # Define the CosineAnnealingWarmRestarts scheduler
# scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=2, eta_min=0.0001)

# # Initialize trackers, these are not parameters and should not be changed
# stale = 0
# best_acc = 0.0

# teacher_model.eval()  # MEDIUM BASELINE
# for epoch in range(n_epochs):

#     # ---------- Training ----------
#     # Make sure the model is in train mode before training.
#     student_model.train()

#     # These are used to record information in training.
#     train_loss = []
#     train_accs = []
#     train_lens = []

#     for batch in tqdm(train_loader):

#         # A batch consists of image data and corresponding labels.
#         imgs, labels = batch
#         imgs = imgs.to(device)
#         labels = labels.to(device)
#         #imgs = imgs.half()
#         #print(imgs.shape,labels.shape)

#         # Forward the data. (Make sure data and model are on the same device.)
#         with torch.no_grad():  # MEDIUM BASELINE
#             teacher_logits = teacher_model(imgs)  # MEDIUM BASELINE

#         logits = student_model(imgs)

#         # Calculate the cross-entropy loss.
#         # We don't need to apply softmax before computing cross-entropy as it is done automatically.
#         loss = loss_fn(logits, labels, teacher_logits) # MEDIUM BASELINE
#         # loss = loss_fn(logits, labels) # SIMPLE BASELINE
#         # Gradients stored in the parameters in the previous step should be cleared out first.
#         optimizer.zero_grad()

#         # Compute the gradients for parameters.
#         loss.backward()

#         # Clip the gradient norms for stable training.
#         grad_norm = nn.utils.clip_grad_norm_(student_model.parameters(), max_norm=cfg['grad_norm_max'])

#         # Update the parameters with computed gradients.
#         optimizer.step()

#         # Update the learning rate
#         scheduler.step()

#         # Compute the accuracy for current batch.
#         # acc = (logits.argmax(dim=-1) == labels).float().sum()
#         # 将 one-hot 编码转换为整数标签索引
#         labels_idx = labels.argmax(dim=-1)  # shape: [64]
#         # 预测类别
#         pred = logits.argmax(dim=-1)  # shape: [64]
#         # 计算准确率
#         acc = (pred == labels_idx).float().sum()  # mean() 比 sum() 更直观

#         # Record the loss and accuracy.
#         train_batch_len = len(imgs)
#         train_loss.append(loss.item() * train_batch_len)
#         train_accs.append(acc)
#         train_lens.append(train_batch_len)

#     train_loss = sum(train_loss) / sum(train_lens)
#     train_acc = sum(train_accs) / sum(train_lens)

#     # Print the information.
#     log(f"[ Train | {epoch + 1:03d}/{n_epochs:03d} ] loss = {train_loss:.5f}, acc = {train_acc:.5f}")

#     # ---------- Validation ----------
#     # Make sure the model is in eval mode so that some modules like dropout are disabled and work normally.
#     student_model.eval()

#     # These are used to record information in validation.
#     valid_loss = []
#     valid_accs = []
#     valid_lens = []

#     # Iterate the validation set by batches.
#     for batch in tqdm(valid_loader):

#         # A batch consists of image data and corresponding labels.
#         imgs, labels = batch
#         imgs = imgs.to(device)
#         labels = labels.to(device)

#         # We don't need gradient in validation.
#         # Using torch.no_grad() accelerates the forward process.
#         with torch.no_grad():
#             logits = student_model(imgs)
#             teacher_logits = teacher_model(imgs) # MEDIUM BASELINE

#         # We can still compute the loss (but not the gradient).
#         loss = loss_fn(logits, labels, teacher_logits) # MEDIUM BASELINE
#         # loss = loss_fn(logits, labels) # SIMPLE BASELINE

#         # Compute the accuracy for current batch.
#         acc = (logits.argmax(dim=-1) == labels).float().sum()

#         # Record the loss and accuracy.
#         batch_len = len(imgs)
#         valid_loss.append(loss.item() * batch_len)
#         valid_accs.append(acc)
#         valid_lens.append(batch_len)
#         #break

#     # The average loss and accuracy for entire validation set is the average of the recorded values.
#     valid_loss = sum(valid_loss) / sum(valid_lens)
#     valid_acc = sum(valid_accs) / sum(valid_lens)

#     # update logs

#     if valid_acc > best_acc:
#         log(f"[ Valid | {epoch + 1:03d}/{n_epochs:03d} ] loss = {valid_loss:.5f}, acc = {valid_acc:.5f} -> best")
#     else:
#         log(f"[ Valid | {epoch + 1:03d}/{n_epochs:03d} ] loss = {valid_loss:.5f}, acc = {valid_acc:.5f}")


#     # save models
#     if valid_acc > best_acc:
#         log(f"Best model found at epoch {epoch}, saving model")
#         torch.save(student_model.state_dict(), f"{save_path}/student_best.ckpt") # only save best to prevent output memory exceed error
#         best_acc = valid_acc
#         stale = 0
#     else:
#         stale += 1
#         if stale > patience:
#             log(f"No improvment {patience} consecutive epochs, early stopping")
#             break
# log("Finish training")
# log_fw.close()

In [ ]:
# Initialize a model, and put it on the device specified.
student_model.to(device)
teacher_model.to(device) # MEDIUM BASELINE

# Initialize optimizer, you may fine-tune some hyperparameters such as learning rate on your own.
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, student_model.parameters()), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=9, T_mult=2, eta_min=1e-5)

# Initialize trackers, these are not parameters and should not be changed
stale = 0
best_acc = 0.0

teacher_model.eval()  # MEDIUM BASELINE
for epoch in range(n_epochs):

    # ---------- Training ----------
    # Make sure the model is in train mode before training.
    student_model.train()

    # These are used to record information in training.
    train_loss = []
    train_loss_hidden = []
    train_accs = []
    train_lens = []
    p = (1+epoch)/n_epochs
    lamb = 1 - p * p # 0-1
    for batch in tqdm(train_loader):
        Slayer1out, Slayer2out, Slayer3out, Tlayer1out, Tlayer2out, Tlayer3out = [], [], [], [], [], []
        # A batch consists of image data and corresponding labels.
        imgs, labels = batch
        imgs = imgs.to(device)
        labels = labels.to(device)
        #imgs = imgs.half()
        #print(imgs.shape,labels.shape)

        # Forward the data. (Make sure data and model are on the same device.)
        with torch.no_grad():  # MEDIUM BASELINE
            teacher_logits = teacher_model(imgs)  # MEDIUM BASELINE


        logits = student_model(imgs)
        slayer1out, slayer2out, slayer3out, tlayer1out, tlayer2out, tlayer3out = \
          Slayer1out[0], Slayer2out[0], Slayer3out[0], Tlayer1out[0], Tlayer2out[0], Tlayer3out[0]
        # Calculate the cross-entropy loss.
        # We don't need to apply softmax before computing cross-entropy as it is done automatically.
        loss_output = loss_fn(logits, labels, teacher_logits) # MEDIUM BASELINE
        loss_hidden = F.smooth_l1_loss(slayer1out, tlayer1out) + F.smooth_l1_loss(slayer2out, tlayer2out) + F.smooth_l1_loss(slayer3out, tlayer3out)
        loss_rkd = loss_fn_rkd(teacher_logits, logits, labels, alpha = 1.0)
        loss_dm = loss_fn_dm(teacher_logits, logits, labels, alpha=0.5)

        loss =  loss_hidden + 10 * lamb * loss_output + loss_rkd + loss_dm
        # Gradients stored in the parameters in the previous step should be cleared out first.
        optimizer.zero_grad()

        # Compute the gradients for parameters.
        loss.backward()

        # Clip the gradient norms for stable training.
        grad_norm = nn.utils.clip_grad_norm_(student_model.parameters(), max_norm=cfg['grad_norm_max'])

        # Update the parameters with computed gradients.
        optimizer.step()

        # Compute the accuracy for current batch.
        # acc = (logits.argmax(dim=-1) == labels).float().sum()
        # 将 one-hot 编码转换为整数标签索引
        labels_idx = labels.argmax(dim=-1)  # shape: [64]
        # 预测类别
        pred = logits.argmax(dim=-1)  # shape: [64]
        # 计算准确率
        acc = (pred == labels_idx).float().sum()  # mean() 比 sum() 更直观

        # Record the loss and accuracy.
        train_batch_len = len(imgs)
        train_loss.append(loss.item() * train_batch_len)
        train_loss_hidden.append(loss_hidden.item() * train_batch_len)
        train_accs.append(acc)
        train_lens.append(train_batch_len)

    train_loss = sum(train_loss) / sum(train_lens)
    train_acc = sum(train_accs) / sum(train_lens)
    train_hidden_loss = sum(train_loss_hidden) / sum(train_lens)

    # Print the information.
    log(f"[ Train | {epoch + 1:03d}/{n_epochs:03d} ] loss = {train_loss:.5f}, hidden_loss = {train_hidden_loss:.5f}, acc = {train_acc:.5f}")


# ---------- Validation ----------
    # Make sure the model is in eval mode so that some modules like dropout are disabled and work normally.
    student_model.eval()

    # These are used to record information in validation.
    valid_loss = []
    valid_accs = []
    valid_lens = []
    # Iterate the validation set by batches.
    for batch in tqdm(valid_loader):

        # A batch consists of image data and corresponding labels.
        imgs, labels = batch
        imgs = imgs.to(device)
        labels = labels.to(device)

        # We don't need gradient in validation.
        # Using torch.no_grad() accelerates the forward process.
        with torch.no_grad():
            logits = student_model(imgs)
            teacher_logits = teacher_model(imgs) # MEDIUM BASELINE

        # We can still compute the loss (but not the gradient).
        loss = loss_fn(logits, labels, teacher_logits) # MEDIUM BASELINE
        # loss = loss_fn(logits, labels) # SIMPLE BASELINE

        # Compute the accuracy for current batch.
        acc = (logits.argmax(dim=-1) == labels).float().sum()

        # Record the loss and accuracy.
        batch_len = len(imgs)
        valid_loss.append(loss.item() * batch_len)
        valid_accs.append(acc)
        valid_lens.append(batch_len)
        #break

    # The average loss and accuracy for entire validation set is the average of the recorded values.
    valid_loss = sum(valid_loss) / sum(valid_lens)
    valid_acc = sum(valid_accs) / sum(valid_lens)

    # update logs

    if valid_acc > best_acc:
        log(f"[ Valid | {epoch + 1:03d}/{n_epochs:03d} ] loss = {valid_loss:.5f}, acc = {valid_acc:.5f} -> best")
    else:
        log(f"[ Valid | {epoch + 1:03d}/{n_epochs:03d} ] loss = {valid_loss:.5f}, acc = {valid_acc:.5f}")


    # save models
    if valid_acc > best_acc:
        log(f"Best model found at epoch {epoch}, saving model")
        torch.save(student_model.state_dict(), f"{save_path}/student_best.ckpt") # only save best to prevent output memory exceed error
        best_acc = valid_acc
        stale = 0
    else:
        stale += 1
        if stale > patience:
            log(f"No improvment {patience} consecutive epochs, early stopping")
            break
log("Finish training")
log_fw.close()

  0%|          | 0/154 [00:00<?, ?it/s]

[ Train | 001/300 ] loss = 27.90672, hidden_loss = 1.74866, acc = 0.20130


  0%|          | 0/54 [00:00<?, ?it/s]

[ Valid | 001/300 ] loss = 1.97722, acc = 0.30700 -> best
Best model found at epoch 0, saving model


  0%|          | 0/154 [00:00<?, ?it/s]

...

  0%|          | 0/154 [00:00<?, ?it/s]

[ Train | 032/300 ] loss = 17.84808, hidden_loss = 1.51703, acc = 0.46459


  0%|          | 0/54 [00:00<?, ?it/s]

[ Valid | 032/300 ] loss = 1.23460, acc = 0.61283


  0%|          | 0/154 [00:00<?, ?it/s]

### Inference
load the best model of the experiment and generate submission.csv

In [ ]:
# create dataloader for evaluation
eval_set = FoodDataset(os.path.join(cfg['dataset_root'], "evaluation"), tfm=test_tfm)
eval_loader = DataLoader(eval_set, batch_size=cfg['batch_size'], shuffle=False, num_workers=0, pin_memory=True)

In [ ]:
# Load model from {exp_name}/student_best.ckpt
student_model_best = get_student_model() # get a new student model to avoid reference before assignment.
ckpt_path = f"{save_path}/student_best.ckpt" # the ckpt path of the best student model.
student_model_best.load_state_dict(torch.load(ckpt_path, map_location='cpu')) # load the state dict and set it to the student model
student_model_best.to(device) # set the student model to device

# Start evaluate
student_model_best.eval()
eval_preds = [] # storing predictions of the evaluation dataset

# Iterate the validation set by batches.
for batch in tqdm(eval_loader):
    # A batch consists of image data and corresponding labels.
    imgs, _ = batch
    # We don't need gradient in evaluation.
    # Using torch.no_grad() accelerates the forward process.
    with torch.no_grad():
        logits = student_model_best(imgs.to(device))
        preds = list(logits.argmax(dim=-1).squeeze().cpu().numpy())
    # loss and acc can not be calculated because we do not have the true labels of the evaluation set.
    eval_preds += preds

def pad4(i):
    return "0"*(4-len(str(i))) + str(i)

# Save prediction results
ids = [pad4(i) for i in range(0,len(eval_set))]
categories = eval_preds

df = pd.DataFrame()
df['Id'] = ids
df['Category'] = categories
df.to_csv(f"{save_path}/submission.csv", index=False) # now you can download the submission.csv and upload it to the kaggle competition.

In [ ]:
# Test Time Augmentation

# Load model from {exp_name}/student_best.ckpt
student_model_best = get_student_model() # get a new student model to avoid reference before assignment.
ckpt_path = f"{save_path}/student_best.ckpt" # the ckpt path of the best student model.
student_model_best.load_state_dict(torch.load(ckpt_path, map_location='cpu')) # load the state dict and set it to the student model
student_model_best.to(device) # set the student model to device

# Start evaluate
student_model_best.eval()

# 5个使用train_tfm测试集
test_loaders = []
for i in range(5):
    test_set_i = FoodDataset(os.path.join(cfg['dataset_root'], "evaluation"), tfm=train_tfm)
    test_loader_i = DataLoader(test_set_i, batch_size=cfg['batch_size'], shuffle=False, num_workers=0, pin_memory=True)
    test_loaders.append(test_loader_i)

# preds存放在6个测试集(1+5)上的测试结果矩阵，每个矩阵是(3347,11)
preds = [[], [], [], [], [], []]
prediction = []
with torch.no_grad():
    # 用test_tfm的测试集
    for data, _ in tqdm(eval_loader):
        test_preds = student_model_best(data.to(device)).cpu().data.numpy()
        preds[0].extend(test_preds)
    # 5个用train_tfm的测试集
    for i, loader in enumerate(test_loaders):
        for data, _ in tqdm(loader):
            test_preds = student_model_best(data.to(device)).cpu().data.numpy()
            preds[i+1].extend(test_preds)


# preds_np = np.array(preds, dtype=object)
preds_np = np.stack(preds, axis=0)  # 变成 shape=(6, 3347, 11)
print('preds_np shape: {}'.format(preds_np.shape))
# 对6个测试结果加权求和
bb = 0.5 * preds_np[0] + 0.1 * preds_np[1] + 0.1 * preds_np[2] + 0.1 * preds_np[3] + 0.1 * preds_np[4] + 0.1 * preds_np[5]
print('bb shape: {}'.format(bb.shape))
prediction = np.argmax(bb, axis=1)

def pad4(i):
    return "0"*(4-len(str(i))) + str(i)

# Save prediction results
ids = [pad4(i) for i in range(0,len(eval_set))]
categories = prediction  # 使用TTA融合后的结果


df = pd.DataFrame()
df['Id'] = ids
df['Category'] = categories
df.to_csv(f"{save_path}/submission_tta_0.csv", index=False) # now you can download the submission.csv and upload it to the kaggle competition.

In [3]:
# ensemble结果

import pandas as pd
from collections import Counter

# 读取多个 CSV 文件
csv_files = ["/content/submission_final.csv", "/content/submission_tta.csv", "/content/submission_tta_0.csv", "/content/submission_tta_1.csv"]

# 读取所有预测结果
preds_list = [pd.read_csv(f).set_index("Id")["Category"] for f in csv_files]

# 创建最终预测 DataFrame
final_preds = preds_list[0].copy()

# 遍历所有 ID 进行投票
for idx in final_preds.index:
    votes = [preds[idx] for preds in preds_list]  # 获取所有模型的预测类别
    final_preds[idx] = Counter(votes).most_common(1)[0][0]  # 选择出现最多的类别

# 保存融合后的结果
final_preds.to_csv("ensemble_voting.csv", index=True, header=["Category"])
print("投票融合完成，结果保存在 ensemble_voting.csv")

投票融合完成，结果保存在 ensemble_voting.csv


> Don't forget to answer the report questions on GradeScope!